In [ ]:
# Hustle Hub – All‑in‑one notebook (splash → login → choices → main app)
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk
import csv
import os

# ============================== PAGE CLASSES ==============================
# (Your existing page classes – slightly adjusted to work as frames)
class UserFeedPage(tk.Frame):
    def __init__(self, parent):
        super().__init__(parent, bg='#EDE3D6')
        self.create_widgets()

    def create_widgets(self):
        # Search bar container
        main_area = tk.Frame(self, bg='#EDE3D6')
        main_area.pack(fill='both', expand=True)

        search_container = tk.Frame(main_area, bg='#EDE3D6')
        search_container.pack(fill='x', pady=40)
        search_center = tk.Frame(search_container, bg='#EDE3D6')
        search_center.pack()
        search_row = tk.Frame(search_center, bg='#EDE3D6')
        search_row.pack()

        tk.Label(search_row, text="⌕", font=('Arial', 20), fg='#B85C2E', bg='#EDE3D6').pack(side='left', padx=(0,10))
        self.search_entry = tk.Entry(search_row, font=('Arial', 16), width=50, bd=2, relief='solid')
        self.search_entry.pack(side='left')
        self.search_entry.bind('<Return>', lambda e: self.search_jobs())
        tk.Label(search_center, text="Browse available Jobs", font=('Segoe UI', 12, 'italic'),
                 fg='#B85C2E', bg='#EDE3D6').pack(pady=(15,0))

        # Scrollable job cards
        canvas = tk.Canvas(main_area, bg='#EDE3D6', highlightthickness=0)
        scrollbar = tk.Scrollbar(main_area, orient='vertical', command=canvas.yview)
        canvas.configure(yscrollcommand=scrollbar.set)
        scrollbar.pack(side='right', fill='y')
        canvas.pack(side='left', fill='both', expand=True)

        jobs_frame = tk.Frame(canvas, bg='#EDE3D6')
        canvas.create_window((0,0), window=jobs_frame, anchor='nw', tags='jobs_frame')
        jobs_frame.bind('<Configure>', lambda e: canvas.configure(scrollregion=canvas.bbox('all')))

        # Sample jobs
        sample = [
            ("Plumber", "Sola's Shack", "2 years exp needed", "2. Akimpemri Cres., Mushin, Lagos", "6h ago", "₦70,000"),
            ("Cleaner", "Royal Ebony College", "None needed", "Radere, Ajah, Lagos", "8h ago", "₦55,000"),
            ("Carpenter", "Dr Clinton Obasanjo", "5 years exp needed", "27 Emerald Avenue, VUC, Lekki", "25m ago", "₦85,000"),
            ("Mechanic", "AutoCare Center", "3 years exp needed", "15 Ikorodu Road, Maryland", "2h ago", "₦80,000"),
            ("Chef", "Tasty Bites Restaurant", "2 years exp needed", "42 Admiralty Way, Lekki", "1d ago", "₦95,000")
        ]
        for title, company, exp, addr, time, salary in sample:
            self.add_job_card(jobs_frame, title, company, exp, addr, time, salary)

    def add_job_card(self, parent, title, company, experience, address, time, salary):
        card = tk.Frame(parent, bg='white', relief='solid', bd=1)
        card.pack(fill='x', pady=10, padx=10)
        top = tk.Frame(card, bg='white')
        top.pack(fill='x', padx=15, pady=(12,5))
        tk.Label(top, text=title, font=('Arial', 18, 'bold'), fg='#1E1E2E', bg='white').pack(side='left')
        tk.Button(top, text="+", font=('Arial', 20, 'bold'), bg='#425D76', fg='#1E1E2E', bd=0, width=3,
                  cursor='hand2', command=lambda t=title: self.apply_for_job(t)).pack(side='right')
        tk.Label(card, text=company, font=('Arial', 14), fg='#333', bg='white').pack(anchor='w', padx=15, pady=2)
        tk.Label(card, text=f"• {experience}", font=('Arial', 12), fg='#666', bg='white').pack(anchor='w', padx=15, pady=2)
        tk.Label(card, text=f"• {address}", font=('Arial', 12), fg='#666', bg='white').pack(anchor='w', padx=15, pady=2)
        bottom = tk.Frame(card, bg='white')
        bottom.pack(fill='x', padx=15, pady=(5,12))
        tk.Label(bottom, text=time, font=('Arial', 11), fg='#888', bg='white').pack(side='left')
        tk.Label(bottom, text=salary, font=('Arial', 12, 'bold'), fg='#425D76', bg='white').pack(side='right')

    def search_jobs(self):
        term = self.search_entry.get()
        if term: messagebox.showinfo("Search", f"Searching for: {term}")
        else: messagebox.showwarning("Search", "Enter a job title or location")

    def apply_for_job(self, title):
        if messagebox.askyesno("Apply", f"Apply for {title}?"):
            messagebox.showinfo("Applied", f"You applied for {title}!")

class MyJobsPage(tk.Frame):
    def __init__(self, parent):
        super().__init__(parent, bg='#f7f9fc')
        self.saved_jobs = []
        self.applied_jobs = []
        self.archived_jobs = []
        self.current_tab = "saved"
        self.create_widgets()

    def create_widgets(self):
        container = tk.Frame(self, bg='#f7f9fc')
        container.place(relx=0.5, rely=0.1, anchor='n')
        tk.Label(container, text="My Jobs", font=("Segoe UI", 28, "bold"), bg='#f7f9fc', fg='#2c3e50').pack(pady=(0,25))
        tab_frame = tk.Frame(container, bg='#f7f9fc')
        tab_frame.pack()
        self.saved_btn = tk.Button(tab_frame, text="Saved", command=lambda: self.switch_tab("saved"),
                                   font=("Segoe UI", 11, "bold"), width=12, padx=10, pady=5)
        self.saved_btn.pack(side='left', padx=8)
        self.applied_btn = tk.Button(tab_frame, text="Applied", command=lambda: self.switch_tab("applied"),
                                     font=("Segoe UI", 11, "bold"), width=12, padx=10, pady=5)
        self.applied_btn.pack(side='left', padx=8)
        self.archived_btn = tk.Button(tab_frame, text="Archived", command=lambda: self.switch_tab("archived"),
                                      font=("Segoe UI", 11, "bold"), width=12, padx=10, pady=5)
        self.archived_btn.pack(side='left', padx=8)
        self.content_frame = tk.Frame(container, bg='#f7f9fc')
        self.content_frame.pack(pady=30)
        self.switch_tab("saved")

    def switch_tab(self, tab):
        self.current_tab = tab
        self.saved_btn.config(bg='#3498db' if tab=="saved" else '#ecf0f1',
                              fg='white' if tab=="saved" else '#34495e')
        self.applied_btn.config(bg='#3498db' if tab=="applied" else '#ecf0f1',
                                fg='white' if tab=="applied" else '#34495e')
        self.archived_btn.config(bg='#3498db' if tab=="archived" else '#ecf0f1',
                                 fg='white' if tab=="archived" else '#34495e')
        for w in self.content_frame.winfo_children(): w.destroy()
        # empty state
        tk.Label(self.content_frame, text="📭", font=("Segoe UI", 48), bg='#f7f9fc', fg='#bdc3c7').pack(pady=(0,15))
        tk.Label(self.content_frame, text="No jobs yet", font=("Segoe UI", 18), bg='#f7f9fc', fg='#95a5a6').pack()
        tk.Label(self.content_frame, text="Start adding jobs to track your applications ✨",
                 font=("Segoe UI", 10), bg='#f7f9fc', fg='#bdc3c7').pack(pady=(10,0))

class MessagesPage(tk.Frame):
    def __init__(self, parent):
        super().__init__(parent, bg='#EDE3D6')
        self.create_widgets()

    def create_widgets(self):
        searchbar = tk.Frame(self, bg="white", height=170)
        searchbar.pack(fill='x')
        searchbar.pack_propagate(False)
        tk.Label(searchbar, text="Messages", bg="white", fg="#1F4E79", font=("Arial", 35, "bold")).place(x=200, y=35)
        # search icon
        try:
            img = Image.open("assets/search.png").resize((25,25))
            icon = ImageTk.PhotoImage(img)
            tk.Label(searchbar, image=icon, bg="white").image = icon
            # place icon – simplified for brevity
        except:
            tk.Label(searchbar, text="🔍", font=("Arial", 16), bg="white").place(x=15, y=75)
        self.search_var = tk.StringVar()
        entry = tk.Entry(searchbar, textvariable=self.search_var, font=("Arial",12), bg="#E6E6E6", width=60)
        entry.place(x=45, y=75, width=1000, height=28)
        entry.bind("<Return>", self.search_messages)
        tk.Button(searchbar, text="Search", command=self.search_messages, bg="#1F4E79", fg="white").place(x=1045, y=75, width=50, height=28)
        filter_frame = tk.Frame(searchbar, bg="white")
        filter_frame.place(x=30, y=115)
        for text, msg in [("All","All messages"),("Unread","No unread"),("Read","Read"),("Starred","Starred"),("Sent","Sent")]:
            btn = tk.Button(filter_frame, text=text, font=('Segoe UI',10), bg='#F0F0F0', fg='#1F4E79', bd=1, relief="solid", padx=15)
            btn.pack(side='left', padx=5, pady=5)
            btn.config(command=lambda t=text, m=msg: messagebox.showinfo(t, m))
        tk.Label(self, text="📬  Your message list will appear here", font=("Segoe UI", 14), bg='#EDE3D6', fg='#555').place(relx=0.5, rely=0.5, anchor='center')

    def search_messages(self, event=None):
        q = self.search_var.get()
        if q: messagebox.showinfo("Search", f"Searching for: {q}")
        else: messagebox.showwarning("Empty Search", "Please enter a search term")

class ProfilePage(tk.Frame):
    def __init__(self, parent):
        super().__init__(parent, bg='#EDE3D6')
        self.create_widgets()

    def create_widgets(self):
        # Header bar
        header = tk.Frame(self, bg='#FFFFFF', height=80)
        header.place(x=0, y=0, relwidth=1, height=80)
        tk.Label(header, text="Personal Profile", font=('Segoe UI',28,'bold'), fg='#1F4E79', bg='#FFFFFF').place(relx=0.55, rely=0.5, anchor='center')

        x_offset = 30
        content_width = self.winfo_screenwidth() - x_offset - 40
        info_width, info_height = 350, 140
        info_y = 130
        info_x = x_offset + (content_width - info_width)//2 - 40
        info_frame = tk.Frame(self, bd=2, relief="solid", bg='#EDE3D6')
        info_frame.place(x=info_x, y=info_y, width=info_width, height=info_height)
        try:
            img = Image.open("assets/prof_pic.png").resize((80,80))
            photo = ImageTk.PhotoImage(img)
            lbl = tk.Label(info_frame, image=photo, bg='#EDE3D6')
            lbl.image = photo
            lbl.place(x=5, y=20)
        except:
            tk.Label(info_frame, text="👤", font=('Segoe UI',60), bg='#EDE3D6').place(x=10, y=20)
        text_frame = tk.Frame(info_frame, bg='#EDE3D6')
        text_frame.place(x=90, y=8)
        for line in ["Username: John Doe","Location: Lagos, Nigeria","Ratings: 4.3/5.0, 2,224 reviews.","Money earned: ₦125,000"]:
            tk.Label(text_frame, text=line, font=('Segoe UI',11), bg='#EDE3D6', fg='#2C3E50').pack(anchor='w', pady=1)

        combined_y = info_y + info_height + 20
        combined_height = 210
        combined_y = self.winfo_screenheight()//2 - combined_height//2 + 40
        combined_frame = tk.Frame(self, bd=2, relief="solid", bg='#EDE3D6')
        combined_frame.place(x=x_offset, y=combined_y, width=content_width, height=combined_height)

        exp_frame = tk.Frame(combined_frame, bg='#EDE3D6')
        exp_frame.place(x=2, y=2, width=content_width-8, height=105)
        tk.Label(exp_frame, text="Experience", font=('Segoe UI',14,'bold'), bg='#EDE3D6', fg='#2C3E50').place(x=20, y=15)
        tk.Label(exp_frame, text="Add your experience to get tailored job recommendations.", font=('Segoe UI',11), bg='#EDE3D6', fg='#555').place(x=20, y=55)

        half = content_width//2
        skills = tk.Frame(combined_frame, bg='#EDE3D6')
        skills.place(x=2, y=102, width=(content_width-8)//2, height=85)
        tk.Label(skills, text="Skills & Languages", font=('Segoe UI',14,'bold'), bg='#EDE3D6', fg='#2C3E50').place(x=20, y=8)
        tk.Button(skills, text="Add a skill or language", font=('Segoe UI',10), bg='#3498DB', fg='white', bd=0, padx=10, pady=5, cursor="hand2",
                  command=lambda: messagebox.showinfo("Add Skill", "Feature coming soon")).place(x=20, y=45)
        cert = tk.Frame(combined_frame, bg='#EDE3D6')
        cert.place(x=2+ (content_width-8)//2, y=102, width=(content_width-8)//2, height=85)
        tk.Label(cert, text="Certifications & License", font=('Segoe UI',14,'bold'), bg='#EDE3D6', fg='#2C3E50').place(x=20, y=8)
        tk.Button(cert, text="Add a certificate or license", font=('Segoe UI',10), bg='#3498DB', fg='white', bd=0, padx=10, pady=5, cursor="hand2",
                  command=lambda: messagebox.showinfo("Add Certificate", "Feature coming soon")).place(x=20, y=45)

        pref_y = combined_y + combined_height + 40
        pref_frame = tk.Frame(self, bd=2, relief="solid", bg='#EDE3D6')
        pref_frame.place(x=x_offset, y=pref_y, width=content_width, height=180)
        tk.Label(pref_frame, text="Job Preferences", font=('Segoe UI',14,'bold'), bg='#EDE3D6', fg='#2C3E50').place(x=20, y=15)
        tk.Label(pref_frame, text="Get customized job recommendations by keeping your preferences updated", font=('Segoe UI',11), bg='#EDE3D6', fg='#555').place(x=20, y=55)
        btn_frame = tk.Frame(pref_frame, bg='#EDE3D6')
        btn_frame.place(x=20, y=100)
        for text, msg in [("Desired Job Title","Set desired job title"),("Desired Work Location","Set work location"),("Desired Minimum Pay","Set desired pay"),("Remote work preferences","Set remote preference")]:
            btn = tk.Button(btn_frame, text=text, font=('Segoe UI',10), bd=1, bg='#ECF0F1', fg='#2C3E50', padx=10, pady=5, cursor="hand2",
                            command=lambda m=msg: messagebox.showinfo("Preference", m))
            btn.pack(side='left', padx=5)

# ============================== SCREEN CLASSES ==============================
class SplashScreen(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, bg='#B05938')
        self.controller = controller
        self.create_widgets()
        self.start_animation()

    def create_widgets(self):
        self.sliding_label = tk.Label(self, bg='#B05938')
        self.sliding_label.place(x=0, y=0, relwidth=1, relheight=1)
        # Load the image for sliding (the same as background)
        img = Image.open("assets/bg_image.png")
        img = img.resize((self.winfo_screenwidth(), self.winfo_screenheight()))
        self.photo = ImageTk.PhotoImage(img)
        self.sliding_label.config(image=self.photo)
        self.sliding_label.image = self.photo
        # Logo, text, button (initially hidden)
        self.logo_img = Image.open("assets/logo.png").resize((300,299))
        self.logo_photo = ImageTk.PhotoImage(self.logo_img)
        self.logo_label = tk.Label(self, image=self.logo_photo, bg='#B05938')
        self.hello_label = tk.Label(self, text="Hey there!", font=('Segoe UI',40,'bold'), fg='#F8F0E3', bg='#B05938')
        self.welcome_label = tk.Label(self, text="Welcome to Hustle Hub", font=('Segoe UI',28,'bold'), fg='#F8F0E3', bg='#B05938')
        self.tagline_label = tk.Label(self, text="Turn your hustle into money now", font=('Segoe Script',20,'italic'), fg='#FFE0C0', bg='#B05938')
        self.next_button = tk.Button(self, text="Get Started", font=('Arial',25,'bold'), bg='#E4D1B9', fg='#B05938', padx=30, pady=10, cursor='hand2', bd=0,
                                     command=lambda: self.controller.show_frame("LoginScreen"))
        # Hide initially
        self.logo_label.place_forget()
        self.hello_label.place_forget()
        self.welcome_label.place_forget()
        self.tagline_label.place_forget()
        self.next_button.place_forget()
        self.anim_y = 0
        self.step = 10
        self.delay = 10

    def start_animation(self):
        if self.anim_y < self.winfo_screenheight():
            self.anim_y += self.step
            self.sliding_label.place(x=0, y=self.anim_y, relwidth=1, relheight=1)
            self.after(self.delay, self.start_animation)
        else:
            # Show UI
            self.logo_label.place(relx=0, x=100, rely=0, y=40)
            self.hello_label.place(relx=0.5, rely=0.25, anchor='center')
            self.welcome_label.place(relx=0.5, rely=0.35, anchor='center')
            self.tagline_label.place(relx=0.5, rely=0.72, anchor='center')
            self.next_button.place(relx=0.5, rely=0.95, y=-50, anchor='s')

class LoginScreen(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, bg='#EDE3D6')
        self.controller = controller
        self.create_widgets()

    def create_widgets(self):
        # Background (full image)
        img = Image.open("assets/bg_image.png").resize((self.winfo_screenwidth(), self.winfo_screenheight()))
        self.bg_photo = ImageTk.PhotoImage(img)
        tk.Label(self, image=self.bg_photo).place(x=0, y=0, relwidth=1, relheight=1)

        logo_img = Image.open("assets/logo.png").resize((300,299))
        self.logo_photo = ImageTk.PhotoImage(logo_img)
        tk.Label(self, image=self.logo_photo, bg='#B05938').place(relx=0, x=100, rely=0, y=40)

        # Frame (login form)
        frame = tk.Frame(self, width=460, height=542, bg="white")
        frame.place(relx=1.0, x=-50, rely=1.0, y=-50, anchor='se')
        frame.pack_propagate(False)
        bg_f = Image.open("assets/frame_bg.png").resize((460,542))
        self.frame_photo = ImageTk.PhotoImage(bg_f)
        tk.Label(frame, image=self.frame_photo).place(x=0, y=0, relwidth=1, relheight=1)

        tk.Label(self, text="Hello!", font=("Kanit",24,"bold"), fg="black", bg="#B05938").place(in_=frame, relx=0.5, y=-70, anchor='s')
        tk.Label(self, text="Welcome back to Hustle Hub", font=("Kanit",24), fg="black", bg="#B05938").place(in_=frame, relx=0.5, y=-20, anchor='s')
        tk.Label(frame, text="Login Account", font=("Arial",22,"bold"), bg="#C0765B").pack(pady=(40,20))
        # Username
        tk.Label(frame, text="Username", font=("Arial",15), bg="#C0765B").pack(anchor='w', padx=20, pady=(0,5))
        self.entry_user = tk.Entry(frame, width=90)
        self.entry_user.pack(anchor='w', padx=20, pady=(0,15))
        # Password
        tk.Label(frame, text="Password", font=("Arial",15), bg="#C0765B").pack(anchor='w', padx=20, pady=(20,5))
        self.entry_pass = tk.Entry(frame, show="*", width=90)
        self.entry_pass.pack(anchor='w', padx=20, pady=(0,30))

        tk.Label(self, text="Hustle Hub, where every skill finds its worth.", font=("Segoe Script",18), fg="black", bg="#DCC8AD").place(x=30, rely=0.5, y=120)
        tk.Button(frame, text="Sign In", font=("Arial",25,"bold"), command=self.sign_in, bg="#B05A39", cursor="hand2", fg="black").place(relx=0.5, rely=0.5, y=40, anchor='center', width=200, height=50)
        tk.Button(frame, text="Create New Account", font=("Arial",12,"bold"), command=lambda: self.controller.show_frame("SignupScreen"), bg="#E4D1B9", cursor="hand2", fg="black").place(relx=0.5, rely=0.5, y=100, anchor='center')

    def sign_in(self):
        uname = self.entry_user.get().strip()
        pwd = self.entry_pass.get().strip()
        if not uname or not pwd:
            messagebox.showerror("Error", "All fields required.")
            return
        file_path = "userinfo.csv"
        if not os.path.exists(file_path):
            messagebox.showerror("Error", "No users registered yet")
            return
        with open(file_path, "r") as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if len(row)>=3 and row[0]==uname and row[2]==pwd:
                    messagebox.showinfo("Success", "Login successful!")
                    self.controller.show_frame("ChoicesScreen")
                    return
        messagebox.showerror("Error", "Invalid username or password")

class SignupScreen(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, bg='#EDE3D6')
        self.controller = controller
        self.create_widgets()

    def create_widgets(self):
        img = Image.open("assets/bg_image.png").resize((self.winfo_screenwidth(), self.winfo_screenheight()))
        self.bg_photo = ImageTk.PhotoImage(img)
        tk.Label(self, image=self.bg_photo).place(x=0, y=0, relwidth=1, relheight=1)

        logo_img = Image.open("assets/logo.png").resize((300,299))
        self.logo_photo = ImageTk.PhotoImage(logo_img)
        tk.Label(self, image=self.logo_photo, bg='#B05938').place(relx=0, x=100, rely=0, y=40)

        frame = tk.Frame(self, width=460, height=542, bg="white")
        frame.place(relx=1.0, x=-50, rely=1.0, y=-50, anchor='se')
        frame.pack_propagate(False)
        bg_f = Image.open("assets/frame_bg.png").resize((460,542))
        self.frame_photo = ImageTk.PhotoImage(bg_f)
        tk.Label(frame, image=self.frame_photo).place(x=0, y=0, relwidth=1, relheight=1)

        tk.Label(self, text="Hello", font=("Kanit",24,"bold"), fg="black", bg="#B05938").place(in_=frame, relx=0.5, y=-70, anchor='s')
        tk.Label(self, text="Welcome to Hustle Hub", font=("Kanit",24), fg="black", bg="#B05938").place(in_=frame, relx=0.5, y=-20, anchor='s')
        tk.Label(frame, text="Create Account", font=("Arial",22,"bold"), bg="#C0765B").pack(pady=(40,20))

        tk.Label(frame, text="Username", font=("Arial",15), bg="#C0765B").pack(anchor='w', padx=20, pady=(0,5))
        self.entry_user = tk.Entry(frame, width=90)
        self.entry_user.pack(anchor='w', padx=20, pady=(0,15))

        tk.Label(frame, text="Email Address", font=("Arial",15), bg="#C0765B").pack(anchor='w', padx=20, pady=(0,5))
        self.entry_email = tk.Entry(frame, width=90)
        self.entry_email.pack(anchor='w', padx=20, pady=(0,15))

        tk.Label(frame, text="Password", font=("Arial",15), bg="#C0765B").pack(anchor='w', padx=20, pady=(0,5))
        self.entry_pass = tk.Entry(frame, show="*", width=90)
        self.entry_pass.pack(anchor='w', padx=20, pady=(0,30))

        tk.Label(self, text="Hustle Hub, where every skill finds its worth.", font=("Segoe Script",18), fg="black", bg="#DCC8AD").place(x=30, rely=0.5, y=120)
        tk.Button(frame, text="Sign Up", font=("Arial",25,"bold"), command=self.sign_up, bg="#B05A39", cursor="hand2", fg="black").place(relx=0.5, rely=0.5, y=80, anchor='center', width=200, height=50)
        tk.Button(frame, text="Go to login", font=("Arial",12,"bold"), command=lambda: self.controller.show_frame("LoginScreen"), bg="#E4D1B9", cursor="hand2", fg="black").place(relx=0.5, rely=0.5, y=140, anchor='center')

    def sign_up(self):
        uname = self.entry_user.get().strip()
        email = self.entry_email.get().strip()
        pwd = self.entry_pass.get().strip()
        if not uname or not email or not pwd:
            messagebox.showerror("Error", "All fields required.")
            return
        file_path = "userinfo.csv"
        exists = os.path.isfile(file_path)
        # check duplicates
        if exists:
            with open(file_path, "r") as f:
                reader = csv.reader(f)
                for row in reader:
                    if len(row)>=2:
                        if row[0]==uname:
                            messagebox.showerror("Error", "Username taken.")
                            return
                        if row[1]==email:
                            messagebox.showerror("Error", "Email registered.")
                            return
        with open(file_path, "a", newline="") as f:
            writer = csv.writer(f)
            if not exists:
                writer.writerow(["username","email","password"])
            writer.writerow([uname, email, pwd])
        messagebox.showinfo("Success", f"Account created for {uname}!")
        self.controller.show_frame("LoginScreen")

class ChoicesScreen(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, bg='white')
        self.controller = controller
        self.create_widgets()

    def create_widgets(self):
        bg_img = Image.open("assets/bg_image.png").resize((self.winfo_screenwidth(), self.winfo_screenheight()))
        self.bg_photo = ImageTk.PhotoImage(bg_img)
        tk.Label(self, image=self.bg_photo).place(x=0, y=0, relwidth=1, relheight=1)

        logo_img = Image.open("assets/logo.png").resize((300,299))
        self.logo_photo = ImageTk.PhotoImage(logo_img)
        tk.Label(self, image=self.logo_photo, bg='#B05938').place(relx=0, x=100, rely=0, y=40)

        tk.Label(self, text="What's your goal?", font=('Arial',35,'bold'), fg='#333333', bg='#B05938').place(relx=0.5, rely=0.3, anchor='center')
        tk.Label(self, text="Choose your role", font=('Arial',26,'bold'), fg='#666666', bg='#DCC8AD').place(relx=0.5, rely=0.7, anchor='center')

        button_frame = tk.Frame(self, bg='#DCC8AD')
        button_frame.place(relx=0.5, rely=0.9, y=-20, anchor='s')
        left = tk.Frame(button_frame, bg='#DCC8AD')
        left.pack(side='left', padx=20)
        right = tk.Frame(button_frame, bg='#DCC8AD')
        right.pack(side='left', padx=20)

        tk.Button(left, text="I AM AN EMPLOYER", font=('Arial',16,'bold'), bg='#B85C2E', fg='white', padx=25, pady=15, cursor='hand2', bd=0,
                  command=lambda: self.controller.show_frame("MainApp")).pack(pady=(0,5))
        tk.Label(left, text="I want to hire workers for my company.", font=('Arial',12), fg='#555555', bg='#DCC8AD').pack()

        tk.Button(right, text="I AM AN EMPLOYEE", font=('Arial',16,'bold'), bg='#B85C2E', fg='white', padx=25, pady=15, cursor='hand2', bd=0,
                  command=lambda: self.controller.show_frame("MainApp")).pack(pady=(0,5))
        tk.Label(right, text="I am looking for jobs and opportunities.", font=('Arial',12), fg='#555555', bg='#DCC8AD').pack()

        tk.Button(self, text="Already have an account? Sign In", font=('Arial',11), fg='#B85C2E', bg='#DCC8AD', activebackground='white', activeforeground='#8a451f', bd=0, cursor='hand2',
                  command=lambda: self.controller.show_frame("LoginScreen")).place(relx=1.0, rely=1.0, x=-20, y=-20, anchor='se')

class MainApp(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, bg='#EDE3D6')
        self.controller = controller
        self.create_sidebar()
        self.container = tk.Frame(self, bg='#EDE3D6')
        self.container.place(x=250, y=0, relwidth=1, relheight=1)
        self.pages = {}
        self.add_page("Home", UserFeedPage)
        self.add_page("My Jobs", MyJobsPage)
        self.add_page("Messages", MessagesPage)
        self.add_page("Profile", ProfilePage)
        self.show_page("Home")

    def create_sidebar(self):
        sidebar = tk.Frame(self, bg='#1E1E2E', width=250, relief='flat')
        sidebar.place(x=0, y=0, width=250, height=self.winfo_screenheight())
        sidebar.pack_propagate(False)
        header = tk.Frame(sidebar, bg='#1E1E2E', height=100)
        header.pack(fill='x', pady=(20,10))
        tk.Label(header, text="Hustle Hub", font=('Segoe UI',18,'bold'), fg='#FFB347', bg='#1E1E2E').pack(pady=10)
        tk.Frame(sidebar, bg='#3A3A4A', height=2).pack(fill='x', padx=20, pady=5)
        self.make_sidebar_button(sidebar, "Home", "🏠")
        self.make_sidebar_button(sidebar, "My Jobs", "💼")
        self.make_sidebar_button(sidebar, "Messages", "💬")
        self.make_sidebar_button(sidebar, "Profile", "👤")
        bottom = tk.Frame(sidebar, bg='#1E1E2E')
        bottom.pack(side='bottom', fill='x', pady=20)
        tk.Button(bottom, text="Logout", font=('Segoe UI',10), bg='#FF6B6B', fg='white', bd=0, padx=10, pady=5, cursor='hand2',
                  command=lambda: self.controller.show_frame("ChoicesScreen")).pack(pady=5)

    def make_sidebar_button(self, parent, text, icon):
        frame = tk.Frame(parent, bg='#1E1E2E')
        frame.pack(fill='x', padx=15, pady=5)
        label = tk.Label(frame, text=f"{icon}  {text}", font=('Segoe UI',12), fg='white', bg='#1E1E2E', anchor='w', cursor="hand2")
        label.pack(fill='x', padx=10, pady=8)
        label.bind('<Button-1>', lambda e: self.show_page(text))

    def add_page(self, name, page_class):
        page = page_class(self.container)
        page.place(x=0, y=0, relwidth=1, relheight=1)
        self.pages[name] = page
        page.lower()

    def show_page(self, name):
        if name in self.pages:
            self.pages[name].lift()
        else:
            messagebox.showerror("Error", f"Page '{name}' not found")

# ============================== CONTROLLER ==============================
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Hustle Hub")
        self.attributes('-fullscreen', True)
        self.bind('<Escape>', lambda e: self.destroy())
        # Container for frames
        container = tk.Frame(self)
        container.pack(side="top", fill="both", expand=True)
        self.frames = {}
        for F in (SplashScreen, LoginScreen, SignupScreen, ChoicesScreen, MainApp):
            frame = F(container, self)
            self.frames[F.__name__] = frame
            frame.place(x=0, y=0, relwidth=1, relheight=1)
        self.show_frame("SplashScreen")

    def show_frame(self, name):
        frame = self.frames.get(name)
        if frame:
            frame.tkraise()
        else:
            messagebox.showerror("Error", f"Screen '{name}' not found")

if __name__ == "__main__":
    app = App()
    app.mainloop()